# DICE ITC 05: Paper Bundle and Reproducibility

This notebook assembles paper-ready figures, reports bootstrap uncertainty, and ends with the reproducibility manifest.


In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone

import hashlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import re
import sys
import tempfile
import types

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
os.environ.setdefault('MPLBACKEND', 'Agg')
for _name in [
    'OPENBLAS_NUM_THREADS',
    'OMP_NUM_THREADS',
    'MKL_NUM_THREADS',
    'NUMEXPR_NUM_THREADS',
    'VECLIB_MAXIMUM_THREADS',
    'BLIS_NUM_THREADS',
]:
    os.environ.setdefault(_name, '1')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Image, Markdown, display
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import average_precision_score, roc_auc_score


In [ ]:
CFG_LABEL = {
    "tier0": "Tier-0",
    "tier0_tier1": "Tier-0/1",
    "tier0_tier1_tier2": "Tier-0/1/2",
}


def _cfg_labels(values: pd.Series) -> list[str]:
    return [CFG_LABEL.get(str(v), str(v)) for v in values]


def render_tier_correlation_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, Path]:
    tier_case = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    tier_final = tier_case[tier_case["config"] == "tier0_tier1_tier2"].copy()
    tier_cols = ["tier0_share", "tier1_alt_share", "tier2_share"]

    tier_corr = tier_final[tier_cols].corr().round(4)
    tier_corr.to_csv(paper_full / "tier_share_correlation.csv")

    stressor_tier = tier_final.groupby("stressor", sort=False)[tier_cols].mean().reset_index()
    stressor_tier.to_csv(paper_full / "stressor_tier_share_summary.csv", index=False)

    tier_final["ternary_x"] = tier_final["tier1_alt_share"] + 0.5 * tier_final["tier2_share"]
    tier_final["ternary_y"] = (np.sqrt(3.0) / 2.0) * tier_final["tier2_share"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    im = axes[0].imshow(tier_corr.values, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    axes[0].set_xticks(range(3), ["Tier-0", "Tier-1", "Tier-2"], rotation=30, ha="right")
    axes[0].set_yticks(range(3), ["Tier-0", "Tier-1", "Tier-2"])
    axes[0].set_title("Tier-share correlation")
    for i in range(3):
        for j in range(3):
            axes[0].text(j, i, f"{tier_corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(stressor_tier))
    axes[1].bar(x, stressor_tier["tier0_share"], label="Tier-0")
    axes[1].bar(x, stressor_tier["tier1_alt_share"], bottom=stressor_tier["tier0_share"], label="Tier-1")
    axes[1].bar(
        x,
        stressor_tier["tier2_share"],
        bottom=stressor_tier["tier0_share"] + stressor_tier["tier1_alt_share"],
        label="Tier-2",
    )
    axes[1].set_xticks(x, stressor_tier["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("Mean tier evidence by stressor")
    axes[1].legend(loc="upper right")

    triangle = np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [0.5, np.sqrt(3.0) / 2.0],
            [0.0, 0.0],
        ]
    )
    axes[2].plot(triangle[:, 0], triangle[:, 1], color="black")
    for stressor, d in tier_final.groupby("stressor", sort=False):
        axes[2].scatter(d["ternary_x"], d["ternary_y"], s=36, alpha=0.8, label=stressor)
    axes[2].text(-0.04, -0.03, "Tier-0")
    axes[2].text(1.01, -0.03, "Tier-1")
    axes[2].text(0.46, np.sqrt(3.0) / 2.0 + 0.03, "Tier-2")
    axes[2].set_title("Per-case tier composition")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
    axes[2].legend(loc="upper right", fontsize=7)

    fig.tight_layout()
    png = paper_fig / "fig_tier_correlation_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_corr, stressor_tier, png


def render_bootstrap_confidence(
    case_pred: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
    samples: int = 1000,
    seed: int = 0,
) -> tuple[pd.DataFrame, Path]:
    rng = np.random.default_rng(seed)
    rows: list[dict[str, float | str]] = []

    for cfg, d in case_pred.groupby("config", sort=False):
        stats: list[dict[str, float]] = []
        for _ in range(samples):
            sample = d.sample(n=len(d), replace=True, random_state=int(rng.integers(1 << 32)))
            if sample["label"].nunique() < 2:
                continue
            benign = sample[sample["label"] == 0]
            anomaly = sample[sample["label"] == 1]
            stats.append(
                {
                    "roc_auc_wc": roc_auc_score(sample["label"], sample["run_score_wc"]),
                    "pr_auc_wc": average_precision_score(sample["label"], sample["run_score_wc"]),
                    "benign_run_false_alarm_rate": benign["run_alert"].mean(),
                    "anomaly_run_detection_rate": anomaly["run_alert"].mean(),
                    "median_time_to_detect_s": anomaly.loc[
                        anomaly["run_alert"] == 1, "time_to_detect_s"
                    ].median(),
                }
            )

        boot = pd.DataFrame(stats)
        rows.append(
            {
                "config": cfg,
                "roc_auc_wc_lo": boot["roc_auc_wc"].quantile(0.025),
                "roc_auc_wc_hi": boot["roc_auc_wc"].quantile(0.975),
                "pr_auc_wc_lo": boot["pr_auc_wc"].quantile(0.025),
                "pr_auc_wc_hi": boot["pr_auc_wc"].quantile(0.975),
                "fpr_lo": boot["benign_run_false_alarm_rate"].quantile(0.025),
                "fpr_hi": boot["benign_run_false_alarm_rate"].quantile(0.975),
                "detect_lo": boot["anomaly_run_detection_rate"].quantile(0.025),
                "detect_hi": boot["anomaly_run_detection_rate"].quantile(0.975),
                "ttd_lo": boot["median_time_to_detect_s"].quantile(0.025),
                "ttd_hi": boot["median_time_to_detect_s"].quantile(0.975),
            }
        )

    out = pd.DataFrame(rows)
    out.to_csv(paper_full / "bootstrap_confidence_intervals.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    x = np.arange(len(out))

    pr_mid = (out["pr_auc_wc_lo"] + out["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - out["pr_auc_wc_lo"], out["pr_auc_wc_hi"] - pr_mid])
    axes[0].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4)
    axes[0].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[0].set_title("Bootstrap AUC-PR CI")

    fpr_mid = (out["fpr_lo"] + out["fpr_hi"]) / 2.0
    fpr_err = np.vstack([fpr_mid - out["fpr_lo"], out["fpr_hi"] - fpr_mid])
    axes[1].errorbar(x, fpr_mid, yerr=fpr_err, fmt="o", capsize=4)
    axes[1].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[1].set_title("Bootstrap benign-FPR CI")

    ttd_mid = (out["ttd_lo"] + out["ttd_hi"]) / 2.0
    ttd_err = np.vstack([ttd_mid - out["ttd_lo"], out["ttd_hi"] - ttd_mid])
    axes[2].errorbar(x, ttd_mid, yerr=ttd_err, fmt="o", capsize=4)
    axes[2].set_xticks(x, out["config"], rotation=30, ha="right")
    axes[2].set_title("Bootstrap time-to-detect CI")

    fig.tight_layout()
    png = paper_fig / "fig_bootstrap_confidence_intervals.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return out, png


def export_llm_case_cards(
    out_full: Path,
    appendix_full: Path,
) -> pd.DataFrame:
    case_diag = pd.read_csv(out_full / "case_diagnosis_summary.csv")
    cards = case_diag[case_diag["config"] == "tier0_tier1_tier2"].copy()
    keep_cols = [
        "case_id",
        "workload",
        "stressor",
        "dominant_tier",
        "dominant_mechanism",
        "tier0_share",
        "tier1_alt_share",
        "tier2_share",
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
        "top_feature_1",
        "top_feature_score_1",
        "top_feature_2",
        "top_feature_score_2",
        "top_feature_3",
        "top_feature_score_3",
        "top_feature_4",
        "top_feature_score_4",
        "top_feature_5",
        "top_feature_score_5",
        "top_mechanism_1",
        "top_mechanism_score_1",
        "top_mechanism_2",
        "top_mechanism_score_2",
        "top_mechanism_3",
        "top_mechanism_score_3",
    ]
    cards = cards[[c for c in keep_cols if c in cards.columns]].copy()

    def _case_card_json(row: pd.Series) -> str:
        payload = {
            "case_id": row.get("case_id"),
            "workload": row.get("workload"),
            "stressor": row.get("stressor"),
            "dominant_tier": row.get("dominant_tier"),
            "dominant_mechanism": row.get("dominant_mechanism"),
            "tier_share": {
                "tier0": row.get("tier0_share"),
                "tier1": row.get("tier1_alt_share"),
                "tier2": row.get("tier2_share"),
            },
            "mechanism_share": {
                "compute": row.get("compute_share"),
                "memory_io": row.get("memory_io_share"),
                "thermal_power": row.get("thermal_power_share"),
                "scheduler_runtime": row.get("scheduler_runtime_share"),
                "platform_pressure": row.get("platform_pressure_share"),
            },
            "top_features": [
                {"name": row.get(f"top_feature_{i}"), "score": row.get(f"top_feature_score_{i}")}
                for i in range(1, 6)
                if pd.notna(row.get(f"top_feature_{i}"))
            ],
            "top_mechanisms": [
                {"name": row.get(f"top_mechanism_{i}"), "score": row.get(f"top_mechanism_score_{i}")}
                for i in range(1, 4)
                if pd.notna(row.get(f"top_mechanism_{i}"))
            ],
        }
        return json.dumps(payload, sort_keys=True)

    cards["diagnostic_case_card_json"] = cards.apply(_case_card_json, axis=1)
    cards["reviewer_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "You are preparing a grounded DICE diagnostic note for a silicon-reliability reviewer. "
            "Use only the supplied case card. Do not invent missing evidence. "
            "Explain the dominant tier, dominant mechanism, and the top residual cues in plain English.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["triage_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Write a compact triage report with five fields: "
            "severity, likely subsystem, evidence summary, two follow-up measurements, and confidence. "
            "If the evidence is weak, say so directly.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards["followup_prompt"] = cards["diagnostic_case_card_json"].apply(
        lambda s: (
            "Use only the supplied DICE case card. Recommend up to three next diagnostic steps. "
            "Each step must cite the specific feature or mechanism that motivated it.\n\n"
            f"Case card JSON:\n{s}"
        )
    )
    cards.to_csv(appendix_full / "llm_case_cards.csv", index=False)
    return cards


def export_llm_diagnostic_model_catalog(appendix_full: Path) -> pd.DataFrame:
    models = pd.DataFrame(
        [
            {
                "model_id": "Qwen/Qwen2.5-7B-Instruct",
                "deployment_role": "primary DICE baseline",
                "priority_rank": 1,
                "params_billions": 7.61,
                "context_tokens": 131072,
                "strengths": "Strong instruction following, structured output behavior, and long-context support.",
                "best_for_dice": "Primary grounded reviewer summaries and structured incident reports from exported case cards.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct",
            },
            {
                "model_id": "microsoft/Phi-4-mini-instruct",
                "deployment_role": "lightweight comparison",
                "priority_rank": 2,
                "params_billions": 3.8,
                "context_tokens": 128000,
                "strengths": "Small footprint, strong reasoning density, and good fit for constrained local diagnostics.",
                "best_for_dice": "Fast first-pass case summaries and follow-up recommendations on a laptop or edge workstation.",
                "source_url": "https://huggingface.co/microsoft/Phi-4-mini-instruct",
            },
            {
                "model_id": "meta-llama/Meta-Llama-3.1-8B-Instruct",
                "deployment_role": "ecosystem baseline",
                "priority_rank": 3,
                "params_billions": 8.0,
                "context_tokens": 128000,
                "strengths": "Broad tooling support, stable chat behavior, and strong general-purpose instruction tuning.",
                "best_for_dice": "Fallback baseline when the deployment stack already supports Llama-family models.",
                "source_url": "https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct",
            },
            {
                "model_id": "Qwen/Qwen2.5-14B-Instruct",
                "deployment_role": "stronger offline review",
                "priority_rank": 4,
                "params_billions": 14.7,
                "context_tokens": 131072,
                "strengths": "Higher-capacity structured reasoning while remaining practical for offline workstation use.",
                "best_for_dice": "Second-pass failure analysis and richer postmortem summaries after the detector has already raised a case.",
                "source_url": "https://huggingface.co/Qwen/Qwen2.5-14B-Instruct",
            },
        ]
    ).sort_values('priority_rank').reset_index(drop=True)
    models.to_csv(appendix_full / "llm_diagnostic_model_catalog.csv", index=False)
    return models


def export_llm_diagnostic_prompt_bundle(
    cards: pd.DataFrame,
    models: pd.DataFrame,
    appendix_full: Path,
) -> pd.DataFrame:
    system_prompt = (
        "You are a DICE diagnostic copilot. You may use only the structured DICE evidence supplied to you. "
        "Do not claim access to raw telemetry, hidden logs, or external knowledge about the run. "
        "If the evidence is incomplete, say that the conclusion is tentative."
    )
    rows = []
    for _, model in models.iterrows():
        for _, card in cards.iterrows():
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "triage",
                    "system_prompt": system_prompt,
                    "user_prompt": card["triage_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "reviewer_summary",
                    "system_prompt": system_prompt,
                    "user_prompt": card["reviewer_prompt"],
                }
            )
            rows.append(
                {
                    "model_id": model["model_id"],
                    "deployment_role": model["deployment_role"],
                    "case_id": card["case_id"],
                    "prompt_type": "followup",
                    "system_prompt": system_prompt,
                    "user_prompt": card["followup_prompt"],
                }
            )

    bundle = pd.DataFrame(rows)
    bundle.to_csv(appendix_full / "llm_diagnostic_prompt_bundle.csv", index=False)
    with (appendix_full / "llm_diagnostic_prompt_bundle.jsonl").open("w") as f:
        for row in bundle.to_dict(orient="records"):
            f.write(json.dumps(row) + "\n")
    return bundle



TIER_ALIAS_MAP = {
    "tier0": ["tier0", "tier-0", "tier 0", "tier-0 evidence", "tier 0 evidence"],
    "tier1_alt": ["tier1", "tier-1", "tier 1", "tier1_alt", "tier-1 evidence", "tier 1 evidence"],
    "tier2": ["tier2", "tier-2", "tier 2", "tier-2 evidence", "tier 2 evidence"],
}

MECHANISM_ALIAS_MAP = {
    "compute": ["compute"],
    "memory_io": ["memory_io", "memory/io", "memory io"],
    "thermal_power": ["thermal_power", "thermal/power", "thermal power"],
    "scheduler_runtime": ["scheduler_runtime", "scheduler runtime"],
    "platform_pressure": ["platform_pressure", "platform pressure"],
}


def _slugify_model_id(model_id: str) -> str:
    return model_id.replace('/', '__').replace('-', '_').replace('.', '_')


def _normalize_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace('_', ' ')
    text = text.replace(':', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def _feature_aliases(name: str | float | None) -> set[str]:
    if pd.isna(name):
        return set()
    name = str(name)
    aliases = {name.lower(), _normalize_text(name)}
    if ':' in name:
        tail = name.split(':', 1)[1]
        aliases.add(tail.lower())
        aliases.add(_normalize_text(tail))
    return {a for a in aliases if a}


def _contains_any(text: str, aliases: set[str] | list[str]) -> bool:
    norm = _normalize_text(text)
    return any(alias and _normalize_text(alias) in norm for alias in aliases)


def run_transformers_llm_batch(
    prompt_bundle: pd.DataFrame,
    appendix_full: Path,
    model_id: str,
    prompt_type: str = "triage",
    max_cases: int = 8,
    max_new_tokens: int = 320,
    temperature: float = 0.0,
) -> pd.DataFrame:
    if importlib.util.find_spec("transformers") is None or importlib.util.find_spec("torch") is None:
        raise RuntimeError("transformers and torch must be installed to run local model inference.")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    subset = prompt_bundle[
        (prompt_bundle["model_id"] == model_id) & (prompt_bundle["prompt_type"] == prompt_type)
    ].copy().head(max_cases)
    if subset.empty:
        raise ValueError(f"No prompts found for model_id={model_id!r} and prompt_type={prompt_type!r}.")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )

    rows = []
    for row in subset.itertuples(index=False):
        messages = [
            {"role": "system", "content": row.system_prompt},
            {"role": "user", "content": row.user_prompt},
        ]
        if hasattr(tokenizer, "apply_chat_template"):
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            text = row.system_prompt + "\n\n" + row.user_prompt

        model_inputs = tokenizer([text], return_tensors="pt")
        model_inputs = {k: v.to(model.device) for k, v in model_inputs.items()}
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0.0,
            temperature=max(temperature, 1e-5),
        )
        new_ids = generated_ids[:, model_inputs["input_ids"].shape[1]:]
        response_text = tokenizer.batch_decode(new_ids, skip_special_tokens=True)[0].strip()
        rows.append(
            {
                "model_id": model_id,
                "case_id": row.case_id,
                "prompt_type": prompt_type,
                "response_text": response_text,
            }
        )

    outputs = pd.DataFrame(rows)
    slug = _slugify_model_id(model_id)
    outputs.to_csv(appendix_full / f"llm_outputs_{slug}_{prompt_type}.csv", index=False)
    with (appendix_full / f"llm_outputs_{slug}_{prompt_type}.jsonl").open("w") as f:
        for rec in outputs.to_dict(orient="records"):
            f.write(json.dumps(rec) + "\n")
    return outputs


def evaluate_llm_grounding_outputs(
    llm_outputs: pd.DataFrame,
    llm_cards: pd.DataFrame,
    appendix_full: Path,
    stem: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    detail = llm_outputs.merge(llm_cards, on="case_id", how="left", suffixes=("", "_card")).copy()

    supported_tier_threshold = 0.05
    supported_mech_threshold = 0.10
    global_mechanisms = list(MECHANISM_ALIAS_MAP.keys())
    global_tiers = list(TIER_ALIAS_MAP.keys())

    rows = []
    for row in detail.itertuples(index=False):
        text = str(row.response_text)
        dominant_tier = getattr(row, "dominant_tier")
        dominant_mechanism = getattr(row, "dominant_mechanism")

        tier_aliases = set(TIER_ALIAS_MAP.get(str(dominant_tier), []))
        dominant_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(dominant_mechanism), [str(dominant_mechanism)]))
        top_feature_aliases = _feature_aliases(getattr(row, "top_feature_1", None))
        top_mech_aliases = set(MECHANISM_ALIAS_MAP.get(str(getattr(row, "top_mechanism_1", '')), [str(getattr(row, "top_mechanism_1", ''))]))

        mentions_dominant_tier = _contains_any(text, tier_aliases)
        mentions_dominant_mechanism = _contains_any(text, dominant_mech_aliases)
        mentions_top_feature_1 = _contains_any(text, top_feature_aliases)
        mentions_top_mechanism_1 = _contains_any(text, top_mech_aliases)

        supported_tiers = {
            "tier0": getattr(row, "tier0_share", 0.0),
            "tier1_alt": getattr(row, "tier1_alt_share", 0.0),
            "tier2": getattr(row, "tier2_share", 0.0),
        }
        unsupported_tier_mentions = any(
            _contains_any(text, TIER_ALIAS_MAP[t]) and supported_tiers.get(t, 0.0) < supported_tier_threshold
            for t in global_tiers
        )

        supported_mechs = {
            "compute": getattr(row, "compute_share", 0.0),
            "memory_io": getattr(row, "memory_io_share", 0.0),
            "thermal_power": getattr(row, "thermal_power_share", 0.0),
            "scheduler_runtime": getattr(row, "scheduler_runtime_share", 0.0),
            "platform_pressure": getattr(row, "platform_pressure_share", 0.0),
        }
        unsupported_mechanism_mentions = any(
            _contains_any(text, MECHANISM_ALIAS_MAP[m]) and supported_mechs.get(m, 0.0) < supported_mech_threshold
            for m in global_mechanisms
        )

        cue_coverage = np.mean([
            float(mentions_dominant_tier),
            float(mentions_dominant_mechanism),
            float(mentions_top_feature_1),
            float(mentions_top_mechanism_1),
        ])

        rows.append(
            {
                "model_id": getattr(row, "model_id"),
                "case_id": getattr(row, "case_id"),
                "prompt_type": getattr(row, "prompt_type"),
                "mentions_dominant_tier": mentions_dominant_tier,
                "mentions_dominant_mechanism": mentions_dominant_mechanism,
                "mentions_top_feature_1": mentions_top_feature_1,
                "mentions_top_mechanism_1": mentions_top_mechanism_1,
                "grounded_core": bool(mentions_dominant_tier and mentions_dominant_mechanism),
                "cue_coverage": cue_coverage,
                "unsupported_tier_mentions": unsupported_tier_mentions,
                "unsupported_mechanism_mentions": unsupported_mechanism_mentions,
                "hallucination_flag": bool(unsupported_tier_mentions or unsupported_mechanism_mentions),
            }
        )

    scored = pd.DataFrame(rows)
    summary = (
        scored.groupby(["model_id", "prompt_type"], sort=False)
        .agg(
            n_outputs=("case_id", "count"),
            grounded_core_rate=("grounded_core", "mean"),
            dominant_tier_rate=("mentions_dominant_tier", "mean"),
            dominant_mechanism_rate=("mentions_dominant_mechanism", "mean"),
            top_feature_1_rate=("mentions_top_feature_1", "mean"),
            top_mechanism_1_rate=("mentions_top_mechanism_1", "mean"),
            mean_cue_coverage=("cue_coverage", "mean"),
            hallucination_rate=("hallucination_flag", "mean"),
        )
        .reset_index()
    )

    summary.to_csv(appendix_full / f"llm_grounding_summary_{stem}.csv", index=False)
    scored.to_csv(appendix_full / f"llm_grounding_detail_{stem}.csv", index=False)
    return summary, scored


def render_paper_performance_stack(
    case_pred: pd.DataFrame,
    overall_full: pd.DataFrame,
    sequential: pd.DataFrame,
    reliability: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    summary = (
        overall_full[
            [
                "config",
                "roc_auc",
                "pr_auc",
                "roc_auc_wc",
                "pr_auc_wc",
                "median_nominal_score_wc",
                "median_anomaly_score_wc",
            ]
        ]
        .merge(
            sequential[
                [
                    "config",
                    "anomaly_detect_rate",
                    "median_time_to_detect_s",
                ]
            ],
            on="config",
        )
        .merge(
            reliability[
                [
                    "config",
                    "target_alpha",
                    "benign_block_false_alarm_rate",
                ]
            ],
            on="config",
        )
    )
    summary["config_label"] = _cfg_labels(summary["config"])
    summary["reliability_margin"] = summary["target_alpha"] - summary["benign_block_false_alarm_rate"]
    summary.to_csv(paper_full / "paper_performance_stack_summary.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    final = case_pred[case_pred["config"] == "tier0_tier1_tier2"].copy()
    benign = final[final["label"] == 0]["run_score_wc"].to_numpy(dtype=float)
    anomaly = final[final["label"] == 1]["run_score_wc"].to_numpy(dtype=float)
    bp = axes[0, 0].boxplot([benign, anomaly], labels=["Benign", "Anomaly"], patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#2563eb", "#dc2626"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    axes[0, 0].set_title("A. Final-head score separation")
    axes[0, 0].set_ylabel("Workload-conditioned run score")

    base_color = "#94a3b8"
    wc_color = "#0f766e"
    for _, row in summary.iterrows():
        axes[0, 1].scatter(row["roc_auc"], row["pr_auc"], color=base_color, s=70)
        axes[0, 1].scatter(row["roc_auc_wc"], row["pr_auc_wc"], color=wc_color, s=90)
        axes[0, 1].annotate(
            row["config_label"],
            (row["roc_auc_wc"], row["pr_auc_wc"]),
            textcoords="offset points",
            xytext=(6, 6),
        )
        axes[0, 1].plot([row["roc_auc"], row["roc_auc_wc"]], [row["pr_auc"], row["pr_auc_wc"]], color="#475569")
    axes[0, 1].set_xlabel("Run-level ROC-AUC")
    axes[0, 1].set_ylabel("Run-level Average Precision")
    axes[0, 1].set_title("B. Digital-twin score refinement")

    x = np.arange(len(summary))
    axes[1, 0].bar(x, summary["benign_block_false_alarm_rate"], color="#f59e0b")
    axes[1, 0].axhline(float(summary["target_alpha"].iloc[0]), color="black", linestyle="--", linewidth=1.2)
    axes[1, 0].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 0].set_ylabel("Empirical benign block FAR")
    axes[1, 0].set_title("C. Calibrated reliability")

    bars = axes[1, 1].bar(x, summary["anomaly_detect_rate"], color="#16a34a", label="Detection rate")
    ax2 = axes[1, 1].twinx()
    ax2.plot(x, summary["median_time_to_detect_s"], color="#1d4ed8", marker="o", linewidth=2, label="Median TTD")
    axes[1, 1].set_xticks(x, summary["config_label"], rotation=30, ha="right")
    axes[1, 1].set_ylabel("Run-level detection rate")
    ax2.set_ylabel("Median time-to-detect (s)")
    axes[1, 1].set_title("D. Operational decision performance")
    axes[1, 1].legend([bars], ["Detection rate"], loc="upper left")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_performance_stack.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return summary, png


def render_explainability_dashboard(
    out_full: Path,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Path]:
    tier_contrib = pd.read_csv(out_full / "stressor_tier_contributions.csv")
    mechanism = pd.read_csv(out_full / "mechanism_group_summary.csv")
    cm = pd.read_csv(out_full / "stressor_confusion_matrix.csv", index_col=0)

    tier_contrib.to_csv(paper_full / "paper_tier_contribution_summary.csv", index=False)
    mechanism.to_csv(paper_full / "paper_mechanism_summary.csv", index=False)
    cm.to_csv(paper_full / "paper_stressor_confusion_matrix.csv")

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    x = np.arange(len(tier_contrib))
    axes[0].bar(x, tier_contrib["tier0_share"], label="Tier-0")
    axes[0].bar(x, tier_contrib["tier1_alt_share"], bottom=tier_contrib["tier0_share"], label="Tier-1")
    axes[0].bar(
        x,
        tier_contrib["tier2_share"],
        bottom=tier_contrib["tier0_share"] + tier_contrib["tier1_alt_share"],
        label="Tier-2",
    )
    axes[0].set_xticks(x, tier_contrib["stressor"], rotation=30, ha="right")
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title("A. Tier contribution by stressor")
    axes[0].legend(loc="upper right")

    mech_cols = [
        "compute_share",
        "memory_io_share",
        "thermal_power_share",
        "scheduler_runtime_share",
        "platform_pressure_share",
    ]
    bottom = np.zeros(len(mechanism))
    colors = ["#0f766e", "#1d4ed8", "#dc2626", "#9333ea", "#b45309"]
    for col, color in zip(mech_cols, colors):
        axes[1].bar(np.arange(len(mechanism)), mechanism[col], bottom=bottom, label=col.replace("_share", ""), color=color)
        bottom += mechanism[col].to_numpy(dtype=float)
    axes[1].set_xticks(np.arange(len(mechanism)), mechanism["stressor"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.0)
    axes[1].set_title("B. Mechanism evidence by stressor")
    axes[1].legend(loc="upper right", fontsize=7)

    im = axes[2].imshow(cm.values, cmap="Blues")
    axes[2].set_xticks(range(len(cm.columns)), list(cm.columns), rotation=30, ha="right")
    axes[2].set_yticks(range(len(cm.index)), list(cm.index))
    axes[2].set_title("C. Stressor diagnosis confusion")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[2].text(j, i, str(int(cm.iloc[i, j])), ha="center", va="center", color="black")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    fig.tight_layout()
    png = paper_fig / "fig_paper_explainability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return tier_contrib, mechanism, cm, png


def render_portability_dashboard(
    frontier: pd.DataFrame,
    holdout: pd.DataFrame,
    bootstrap_ci: pd.DataFrame,
    paper_full: Path,
    paper_fig: Path,
) -> tuple[pd.DataFrame, Path]:
    view = frontier.copy()
    view["config_label"] = _cfg_labels(view["config"])
    holdout_view = holdout.copy()
    holdout_view["config_label"] = _cfg_labels(holdout_view["config"])
    boot_view = bootstrap_ci.copy()
    boot_view["config_label"] = _cfg_labels(boot_view["config"])

    portability_summary = view[
        [
            "config",
            "config_label",
            "n_features",
            "portable_pr_auc",
            "holdout_worst_pr_auc",
            "reliability_margin",
            "joint_detection_diagnosis",
        ]
    ].copy()
    portability_summary.to_csv(paper_full / "paper_portability_summary.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    scatter = axes[0].scatter(
        view["n_features"],
        view["portable_pr_auc"],
        s=view["joint_detection_diagnosis"].fillna(0.0) * 1800 + 140,
        c=view["reliability_margin"],
        cmap="viridis",
        edgecolor="black",
        linewidth=0.8,
    )
    for _, row in view.iterrows():
        axes[0].annotate(row["config_label"], (row["n_features"], row["portable_pr_auc"]), textcoords="offset points", xytext=(6, 6))
    axes[0].set_xlabel("Median active features")
    axes[0].set_ylabel("Portable AUC-PR")
    axes[0].set_title("A. Observability-portability frontier")
    fig.colorbar(scatter, ax=axes[0], fraction=0.046, pad=0.04)

    x = np.arange(len(holdout_view))
    width = 0.35
    axes[1].bar(x - width / 2.0, holdout_view["mean_pr_auc"], width=width, label="Mean holdout PR")
    axes[1].bar(x + width / 2.0, holdout_view["worst_pr_auc"], width=width, label="Worst holdout PR")
    axes[1].set_xticks(x, holdout_view["config_label"], rotation=30, ha="right")
    axes[1].set_ylim(0.0, 1.05)
    axes[1].set_title("B. Holdout portability")
    axes[1].legend(loc="upper right")

    x = np.arange(len(boot_view))
    pr_mid = (boot_view["pr_auc_wc_lo"] + boot_view["pr_auc_wc_hi"]) / 2.0
    pr_err = np.vstack([pr_mid - boot_view["pr_auc_wc_lo"], boot_view["pr_auc_wc_hi"] - pr_mid])
    axes[2].errorbar(x, pr_mid, yerr=pr_err, fmt="o", capsize=4, color="#1d4ed8", label="AP CI")
    det_mid = (boot_view["detect_lo"] + boot_view["detect_hi"]) / 2.0
    det_err = np.vstack([det_mid - boot_view["detect_lo"], boot_view["detect_hi"] - det_mid])
    axes[2].errorbar(x, det_mid, yerr=det_err, fmt="o", capsize=4, color="#16a34a", label="Detect-rate CI")
    axes[2].set_xticks(x, boot_view["config_label"], rotation=30, ha="right")
    axes[2].set_ylim(0.0, 1.05)
    axes[2].set_title("C. Bootstrap uncertainty")
    axes[2].legend(loc="lower right")

    fig.tight_layout()
    png = paper_fig / "fig_paper_portability_dashboard.png"
    fig.savefig(png, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return portability_summary, png


In [ ]:
def _looks_like_repo_root(base: Path) -> bool:
    return (base / 'data generation').exists() and (base / 'environment.yml').exists()


def resolve_repo_root(start: Path) -> Path:
    candidates = []
    seen = set()

    def add_candidate(p: Path | None) -> None:
        if p is None:
            return
        try:
            rp = p.expanduser().resolve()
        except FileNotFoundError:
            rp = p.expanduser()
        key = str(rp)
        if key not in seen:
            seen.add(key)
            candidates.append(rp)

    add_candidate(start)
    for base in [start, *start.parents]:
        add_candidate(base)

    env_hint = os.environ.get('DICE_REPO_ROOT')
    if env_hint:
        add_candidate(Path(env_hint))

    home = Path.home()
    for base in [
        home / 'Documents' / 'New project' / 'DICE',
        home / 'Documents' / 'New project' / 'DICE-latest-sync',
        home / 'DICE',
        home / 'Downloads' / 'DICE',
        home / 'Downloads' / 'DICE-latest-sync',
    ]:
        add_candidate(base)

    for root in [home, home / 'Documents', home / 'Documents' / 'New project', home / 'Downloads']:
        if root.exists():
            for child in root.iterdir():
                if child.is_dir() and 'dice' in child.name.lower():
                    add_candidate(child)

    for base in candidates:
        if _looks_like_repo_root(base):
            return base

    raise RuntimeError(
        'Could not locate the DICE repository root. Launch the notebook from a DICE checkout or set DICE_REPO_ROOT.'
    )


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## 12. Uncertainty and Confidence Intervals

The deployed DICE head remains lightweight.
Extra offline runtime is spent here on bootstrap confidence intervals so the reported paper metrics are better defended.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')


In [ ]:
BOOTSTRAP_SAMPLES = 1000
bootstrap_ci, bootstrap_png = render_bootstrap_confidence(case_pred, PAPER_FULL, PAPER_FIG, samples=BOOTSTRAP_SAMPLES, seed=0)
print('Bootstrap confidence intervals')
display(bootstrap_ci)
display(Image(filename=str(bootstrap_png)))


## Quick Jump: Paper Figures

If you only want the reviewer-facing paper artifacts, start here once the result CSVs already exist:
- `## 13. Paper-Ready Figure Bundle`
- then review the claim-boundary and reproducibility sections below.


## 13. Paper-Ready Figure Bundle

This section assembles the reviewer-facing composite figures used in the main paper and appendix. It is the fastest place to start if the result CSVs already exist and you only need paper-ready visual artifacts.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')

if (PAPER_FULL / 'conformal_reliability_summary.csv').exists():
    reliability_bundle = pd.read_csv(PAPER_FULL / 'conformal_reliability_summary.csv')
else:
    reliability_bundle = reliability.copy() if 'reliability' in globals() else pd.DataFrame()

performance_stack, performance_stack_png = render_paper_performance_stack(
    case_pred,
    overall_full,
    sequential,
    reliability_bundle,
    PAPER_FULL,
    PAPER_FIG,
)

tier_summary, mechanism_summary, cm_summary, explainability_png = render_explainability_dashboard(
    OUT_FULL,
    PAPER_FULL,
    PAPER_FIG,
)

if 'frontier' not in globals():
    diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
    frontier = (
        overall_full[['config', 'roc_auc_wc', 'pr_auc_wc']]
        .merge(
            sequential[['config', 'benign_run_alert_rate', 'anomaly_detect_rate', 'median_time_to_detect_s']],
            on='config',
            how='left',
        )
        .merge(
            diagnosis[['config', 'top1_acc', 'top2_acc']],
            on='config',
            how='left',
        )
    )
    if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
        holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
        frontier = frontier.merge(
            holdout[['config', 'mean_pr_auc', 'worst_pr_auc']],
            on='config',
            how='left',
        )
        frontier['portable_pr_auc'] = frontier['mean_pr_auc'].fillna(frontier['pr_auc_wc'])
    else:
        holdout = pd.DataFrame()
        frontier['portable_pr_auc'] = frontier['pr_auc_wc']
else:
    if 'holdout' not in globals() or holdout is None:
        holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv') if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists() else pd.DataFrame()

if 'bootstrap_ci' not in globals():
    bootstrap_ci, bootstrap_png = render_bootstrap_confidence(case_pred, PAPER_FULL, PAPER_FIG, samples=1000, seed=0)

portability_summary, portability_png = render_portability_dashboard(
    frontier,
    holdout,
    bootstrap_ci,
    PAPER_FULL,
    PAPER_FIG,
)

print('Performance stack summary')
display(performance_stack)
print('Portability summary')
display(portability_summary)
display(Image(filename=str(performance_stack_png)))
display(Image(filename=str(explainability_png)))
display(Image(filename=str(portability_png)))


## 14. Research Directions: Digital Twins + LLMs

The papers below are good next-step inspirations for DICE. They are not claims of what DICE already does; they point to concrete directions that would make the next version of the work stronger.

1. **Sequential conformal for time series**: Xu and Xie, *Sequential Predictive Conformal Inference for Time Series* (ICML 2023).
   Use this to move DICE from split-conformal under exchangeability toward adaptive sequential calibration under temporal dependence.
   Link: https://proceedings.mlr.press/v202/xu23r.html

2. **Fleet-scale / federated twins**: San, Pawar, and Rasheed, *Decentralized digital twins of complex dynamical systems* (Scientific Reports 2023).
   Use this to turn the current optional-fleet story into a real decentralized or federated DICE update path.
   Link: https://www.nature.com/articles/s41598-023-47078-9

3. **World-model direction for twins**: Zhou et al., *Digital Twin AI: Opportunities and Challenges from Large Language Models to World Models* (arXiv 2026).
   Use this as the framing for evolving DICE from a behavioral micro-twin into a richer world-model twin while staying honest about what the current paper implements.
   Link: https://arxiv.org/abs/2601.01321

4. **Generative trajectory twins**: Makarov et al., *Large language models forecast patient health trajectories enabling digital twins* (npj Digital Medicine 2025).
   The important idea is not the medical domain; it is the combination of trajectory forecasting, preserved cross-correlations, and explainability. That is relevant if DICE ever moves from anomaly scoring toward generative telemetry forecasting.
   Link: https://www.nature.com/articles/s41746-025-02004-3

5. **Local LLM-enabled twin reasoning**: Lammert et al., *Large language models-enabled digital twins for precision medicine in rare gynecological tumors* (npj Digital Medicine 2025).
   This is a good analogue for a local DICE copilot that reasons over structured case cards, recommends follow-up diagnostics, and keeps sensitive data local.
   Link: https://www.nature.com/articles/s41746-025-01810-z

6. **Edge-aware LLM plus twin systems**: Hong, Wu, and Morello, *LLM-Twin* (Scientific Reports 2024).
   The useful idea here is the mini-giant deployment split. For DICE, that suggests a tiny local model for case summarization plus an optional larger offline model for deeper review.
   Link: https://www.nature.com/articles/s41598-024-69474-5

7. **Self-checking diagnostic summaries**: Liu et al., *Large Language Models have Intrinsic Self-Correction Ability* (arXiv 2024).
   If you build an LLM-based DICE copilot, this supports adding a second-pass self-check to keep the report conservative and grounded.
   Link: https://arxiv.org/abs/2406.15673

The strongest next-step research directions for DICE are therefore:
- sequential conformal recalibration under temporal drift,
- federated or decentralized edge-package refresh,
- richer trajectory-level twins that preserve cross-feature structure,
- a local LLM copilot that only reads structured DICE evidence rather than raw telemetry.


## 15. Paper Claim Boundaries

This notebook is designed to support a clear and defensible paper narrative. The main claim boundaries are as follows.

- **Not a full virtual replica.** DICE is a behavioral micro-twin that maintains a virtual benign reference in telemetry space.
- **Not an LLM-based detector.** The detector is the digital twin plus residual scoring, conformal calibration, and persistence.
- **Not a synthesized hardware implementation.** The accelerator section reports a projected score-stage complexity estimate only.
- **Yes to calibrated detection and diagnosis.** The main supported claims are tier-aware behavioral digital twinning, false-alarm-controlled block decisions, diagnosis cues, observability-budget analysis, and a plausible co-design path.


## 16. Reproducibility Manifest

This final cell confirms the dataset hash, environment hash, and notebook runtime summary so the released analysis can be checked across machines.


In [ ]:
manifest = json.loads(MANIFEST.read_text())
print('Manifest path:', MANIFEST)
print('Dataset SHA256:', manifest['dataset_digest']['sha256'])
print('Environment file SHA256:', manifest['environment_files']['environment_yml']['sha256'])
print('Requirements SHA256:', manifest['environment_files']['requirements_txt']['sha256'])
if NOTEBOOK_RUNTIME.exists():
    runtime_summary = json.loads(NOTEBOOK_RUNTIME.read_text())
    print('Notebook runtime summary:', runtime_summary)
print('Main paper artifacts:', PAPER_FULL)
print('Appendix artifacts :', APPENDIX_FULL)
